## FACT ORDERS

#### Data Reading

In [ ]:
df = spark.sql(
"""
    SELECT * FROM learn_e2e_1.silver.orders
""")



In [26]:
%sql

SELECT count(*) from learn_e2e_1.silver.orders

,count(*)
0,20000


In [20]:
df_dimcus = spark.sql(
    """
        SELECT DimCustomerKey, customer_id as dim_customer_id FROM learn_e2e_1.gold.customers
    """
)
df_dimprod = spark.sql(
    """
        SELECT product_id as DimProductKey, product_id as dim_product_id FROM learn_e2e_1.gold.dimproducts
    """
)

#### Fact Dataframe

In [21]:
df_fact = df.join(df_dimcus, df["customer_id"] == df_dimcus["dim_customer_id"], how="left") \
    .join(df_dimprod, df["product_id"] == df_dimprod["dim_product_id"], how="left") \
    .drop("dim_customer_id", "dim_product_id", "customer_id", "product_id") 

In [22]:
df_fact

,order_id,order_date,quantity,total_amount,year,month,day,DimCustomerKey,DimProductKey
0,O00001,2023-03-22,3,2022.87,2023,3,22,963,P0159
1,O00002,2023-06-30,2,3560.74,2023,6,30,1631,P0036
2,O00003,2023-11-06,3,5903.52,2023,11,6,1416,P0427
3,O00004,2024-02-27,3,4107.99,2024,2,27,1723,P0332
4,O00005,2024-10-13,5,5784.95,2024,10,13,1232,P0038
5,O00006,2023-05-17,5,407.75,2023,5,17,1768,P0174
6,O00007,2024-01-18,4,4907.64,2024,1,18,337,P0352
7,O00008,2023-01-10,4,7037.88,2023,1,10,1738,P0172
8,O00009,2023-04-20,3,4076.97,2023,4,20,1924,P0238
9,O00010,2023-07-07,4,5695.64,2023,7,7,1733,P0258


#### Upsert on Fact Table

In [25]:
from delta.tables import DeltaTable

if spark.catalog.tableExists("learn_e2e_1.gold.FactOrders"):
    dlt_obj = DeltaTable.forName(spark, "learn_e2e_1.gold.FactOrders") \
        .alias("tgt") \
        .merge(df_fact.alias("src"), \
            "tgt.order_id = src.order_id \
            AND tgt.DimCustomerKey = src.DimCustomerKey \
            AND tgt.DimProductKey = src.DimProductKey") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
        
else:
    df_fact.write.format("delta") \
        .option("path", "s3://learn-databricks-project-e2e-1-gold/factOrders") \
        .saveAsTable("learn_e2e_1.gold.FactOrders")
        
        
        

UnsupportedOperationException: [DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE] Cannot perform Merge as multiple source rows matched and attempted to modify the same
target row in the Delta table in possibly conflicting ways. By SQL semantics of Merge,
when multiple source rows match on the same target row, the result may be ambiguous
as it is unclear which source row should be used to update or delete the matching
target row. You can preprocess the source table to eliminate the possibility of
multiple matches. Please refer to
https://docs.databricks.com/delta/merge.html#merge-error

JVM stacktrace:
com.databricks.sql.transaction.tahoe.DeltaUnsupportedOperationException
	at com.databricks.sql.transaction.tahoe.DeltaErrorsEdge.multipleSourceRowMatchingTargetRowInMergeException(DeltaErrorsEdge.scala:821)
	at com.databricks.sql.transaction.tahoe.DeltaErrorsEdge.multipleSourceRowMatchingTargetRowInMergeException$(DeltaErrorsEdge.scala:816)
	at com.databricks.sql.transaction.tahoe.D

In [24]:
%sql

SELECT count(*) from learn_e2e_1.gold.FactOrders

,count(*)
0,20000
